# PySpark Fundamentals

An approachable, hands-on guide to the topics in the [original PySpark course notebook](https://github.com/anshlambagit/PySpark-Full-Course/blob/main/DATA%20and%20NOTEBOOK/1_Tutoral.ipynb).

Each topic has a short explanation followed immediately by a code example. The examples use small in-memory datasets wherever practical, so you can run the notebook from top to bottom without downloading course data.

## Contents

- Reading JSON
- Data-reading utilities
- Schema definition with DDL
- Schema definition with StructType
- SELECT
- Aliases
- FILTER
- Renaming columns with withColumnRenamed
- Adding and replacing columns with withColumn
- Type casting
- Sorting
- LIMIT
- Dropping columns
- Dropping duplicate rows
- UNION
- UNION BY NAME
- String functions
- Current date
- Date add and subtract
- Date difference
- Date formatting
- Handling null values
- SPLIT and indexing
- EXPLODE
- GROUP BY
- COLLECT_LIST
- PIVOT
- WHEN and OTHERWISE
- JOINS
- ROW_NUMBER
- RANK versus DENSE_RANK
- Cumulative sum
- User-defined functions (UDF)
- Writing CSV and save modes
- Writing Parquet
- Writing a table
- Spark SQL



## Setup

This first code cell imports the tools used throughout the notebook, starts Spark, and creates a small employee dataset that later examples reuse.

### Imports

- `import json` loads Python's standard JSON module. The JSON-reading lesson uses `json.dumps()` to turn Python dictionaries into JSON-formatted strings before Spark reads them.
- `import tempfile` provides temporary directories. The file-writing examples use them so they can demonstrate output without leaving data files in your project.
- `from pathlib import Path` gives a convenient, cross-platform way to join directory and file names, such as `Path(temp_dir) / "employees_csv"`.
- `from pyspark.sql import SparkSession, Window` imports the entry point for creating or accessing a Spark SQL session and the builder for window specifications.
- `functions as F` imports Spark's built-in SQL functions under the short alias `F`. For example, `F.col()` creates a Spark column expression and `F.sum()` creates an aggregate expression. These are Spark expressions, not ordinary Python functions applied to each row.
- `types as T` imports Spark SQL data types and schema classes, such as `T.IntegerType()` and `T.StructType()`.

### Starting Spark

`SparkSession` is the main entry point for working with Spark DataFrames and Spark SQL. The variable `spark` is the session used by the examples to create DataFrames, read and write data, and run SQL.

The chained builder calls configure that session:

1. `SparkSession.builder` starts a session builder, which collects settings before a session is created.
2. `.master("local[*]")` selects the execution mode. `local` runs on this computer rather than connecting to a cluster; `[*]` asks Spark to use the available local processor cores. For a cluster deployment, the master is normally configured by the cluster environment instead.
3. `.appName("PySpark Fundamentals")` sets a human-readable application name, useful in Spark's UI and logs.
4. `.config("spark.sql.shuffle.partitions", "2")` sets a Spark SQL configuration. A shuffle redistributes data between tasks, often for grouping or joining. The smaller partition count is suitable for this tiny local teaching dataset; it is not a general production tuning recommendation.
5. `.getOrCreate()` returns an existing session if one is already active in this Python process, or creates a new one using the builder settings if needed. This avoids creating competing sessions when rerunning notebook cells.

### Defining the example data and schema

`employee_rows` is a Python list of tuples. Each tuple is one employee record, and values must line up with the column order in the schema. `None` represents a missing value (Spark SQL `NULL`).

`employee_schema` is a DDL schema string. Each comma-separated entry names a column and its Spark SQL type: for example, `id INT` declares an integer column and `name STRING` declares text. Supplying a schema tells Spark how to interpret each position instead of asking it to infer types.

`spark.createDataFrame(employee_rows, schema=employee_schema)` converts those local records into a Spark DataFrame. A DataFrame is a distributed, named-column table with a schema; Spark operations build a query plan that is evaluated when an action such as `show()` requests results.

`employees.show(truncate=False)` is an action that displays the rows. `truncate=False` prevents long text values from being shortened in the display. `show()` is for inspection; it does not change the DataFrame.

### Reading a schema

Every Spark DataFrame has a schema: its column names, data types, and nullability metadata. `employees.printSchema()` prints this structure in a tree-like form. For example, it lets you check that `salary` is a `double` and `join_date` is still a `string`. It shows metadata, not the row values; use `show()` for sample values.

### Environment prerequisite

If PySpark is not installed in the selected notebook environment, run `%pip install pyspark`, restart the kernel, and then execute the notebook from the first code cell. Spark also needs a compatible Java runtime. In a managed cluster or cloud notebook, use the environment's existing Spark setup instead of starting another local installation.

In [ ]:
import json
import tempfile
from pathlib import Path

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("PySpark Fundamentals")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

employee_rows = [
    (1, "alice johnson", "Data", 29, 92000.0, "New York", "python,spark", "2022-03-15"),
    (2, "bob smith", "Data", 35, 87000.0, "Boston", "sql,python", "2021-07-01"),
    (3, "cara ng", "BI", None, 79000.0, "Boston", "sql,tableau", "2023-01-10"),
    (4, "david lee", "Operations", 42, 110000.0, "Chicago", "excel,sql", "2019-11-20"),
    (5, "eva brown", None, 31, 88000.0, "New York", "python,tableau", "2022-08-05"),
    (6, "frank green", "Data", 26, 72000.0, "Seattle", "spark,python", "2024-02-12"),
]

employee_schema = (
    "id INT, name STRING, department STRING, age INT, salary DOUBLE, "
    "city STRING, skills STRING, join_date STRING"
)
employees = spark.createDataFrame(employee_rows, schema=employee_schema)
employees.show(truncate=False)


## Reading JSON

JSON (JavaScript Object Notation) stores records as text objects with named keys and values. Spark's JSON reader can convert newline-delimited JSON records into columns: keys become column names, and Spark can infer data types or use a schema you provide.

### What the example does

1. `json.dumps({...})` serializes each Python dictionary into a valid JSON string. `json` was imported in the setup cell for this step.
2. `parallelize(json_records)` makes a small local Spark RDD from the strings. This is only a convenient way to demonstrate reading without creating an input file; in a real project, prefer `spark.read.json("/path/to/file.json")` or a storage URI.
3. `spark.read.json(json_rdd)` parses the JSON and returns a DataFrame. Here Spark infers `id` as an integer, `name` as a string, and `active` as a Boolean.
4. `printSchema()` confirms the inferred columns and types; `show()` displays the parsed records.

JSON input files commonly contain one JSON object per line. If records have inconsistent or evolving shapes, explicitly supplying a schema can make the resulting columns and types predictable.

In [ ]:
json_records = [
    json.dumps({"id": 101, "name": "Mina", "active": True}),
    json.dumps({"id": 102, "name": "Omar", "active": False}),
]
json_rdd = spark.sparkContext.parallelize(json_records)
json_df = spark.read.json(json_rdd)
json_df.printSchema()
json_df.show()


## Data-reading utilities

This example introduces the most useful ways to inspect a DataFrame. These calls answer different questions:

- `printSchema()` displays column names, Spark SQL types, and nullability metadata. It does not scan the data to print rows.
- `show(3, truncate=False)` displays at most three rows. The `3` controls the row count; `truncate=False` asks the notebook output to show full cell values.
- `count()` is an action that computes the total row count and returns an integer to Python. Counting can require Spark to process the dataset.
- `first()` returns the first row as a Python-side Spark `Row` object, or `None` if the DataFrame is empty.
- `take(2)` returns at most two `Row` objects to the driver. This is useful for a small preview.

Actions such as `show()`, `count()`, `first()`, and `take()` cause Spark to execute the plan needed to answer them. Be cautious with `collect()`: it transfers every row to the Python driver and can exhaust its memory for a large dataset. Keep full-data processing distributed in Spark.

In [ ]:
employees.printSchema()
employees.show(3, truncate=False)
print("Row count:", employees.count())
print("First row:", employees.first())
print("First two rows:", employees.take(2))


## Schema definition with DDL

A schema defines the names and data types of a DataFrame's columns. Here the DDL (data definition language) string `"id INT, name STRING, active BOOLEAN"` describes three fields using Spark SQL type names.

`spark.read.schema(json_schema)` attaches that expected schema to a reader. The following `.json(json_rdd)` parses the same JSON records as the previous lesson but uses the declared types rather than inferring them. `printSchema()` verifies what was declared, and `show()` lets you inspect parsed values.

Explicit schemas are valuable because inference may require Spark to inspect data and can be unreliable when values are missing or inconsistent. The input must still be compatible with the declared types; a schema is not a substitute for checking malformed records. DDL strings are compact and readable, while `StructType` is useful when you need Python objects, nullability details, or nested schemas.

In [ ]:
json_schema = "id INT, name STRING, active BOOLEAN"
typed_json_df = spark.read.schema(json_schema).json(json_rdd)
typed_json_df.printSchema()
typed_json_df.show()


## Schema definition with StructType

`StructType` is Spark's Python representation of a record schema, and each `StructField` defines one named field. This form can express details such as each field's Spark type and whether it is allowed to be null.

In this example:

- `T.StructType([...])` creates a schema containing three fields.
- `T.StructField("id", T.IntegerType(), nullable=False)` declares a non-null integer identifier.
- The `name` field is likewise a required string.
- `active` is a Boolean whose `nullable=True` metadata permits a missing value.
- `spark.createDataFrame(..., schema=person_schema)` assigns that schema to the supplied tuples. The tuple order must match the field order.
- `printSchema()` shows the resulting types and nullable flags; `show()` prints the records, including `NULL` for `None`.

Use Spark's types from `pyspark.sql.types`; they describe Spark SQL data, not Python type annotations. For nested JSON, a `StructType` can itself contain other `StructType`, `ArrayType`, or `MapType` fields.

In [ ]:
person_schema = T.StructType(
    [
        T.StructField("id", T.IntegerType(), nullable=False),
        T.StructField("name", T.StringType(), nullable=False),
        T.StructField("active", T.BooleanType(), nullable=True),
    ]
)
people = spark.createDataFrame(
    [(201, "Ira", True), (202, "Jo", None)],
    schema=person_schema,
)
people.printSchema()
people.show()


## SELECT

Projection means choosing which columns or calculated expressions appear in a result. `DataFrame.select()` creates a new DataFrame with that projection; it does not mutate `employees`.

The first call selects the existing `name`, `department`, and `salary` columns. The second demonstrates a calculated projection: `F.col("salary")` refers to the Spark column named `salary`, multiplying it by `1.1` builds a Spark expression for a 10% increase, and `.alias(...)` gives that expression a readable output name. `id` is included alongside it so each result can be associated with an employee.

`F.col()` is useful when building expressions; plain strings also work for simple column selection. These operations are evaluated by Spark across the data, rather than by looping over rows in Python.

In [ ]:
employees.select("name", "department", "salary").show()
employees.select(
    "id",
    (F.col("salary") * 1.1).alias("salary_with_10_percent_raise"),
).show()


## Aliases

An alias is an alternate name for a column or expression in the output schema. `Column.alias("new_name")` changes the selected result's label; it does not rename the source DataFrame or alter its stored data.

Here `F.col("name")` and `F.col("salary")` refer to existing columns, and their aliases produce `employee_name` and `annual_salary` in the selected DataFrame. Aliases make reports self-explanatory and are particularly useful for calculated expressions or for distinguishing columns with identical names after a join.

In [ ]:
employees.select(
    F.col("name").alias("employee_name"),
    F.col("salary").alias("annual_salary"),
).show()


## FILTER

Filtering selects rows that satisfy a condition. `DataFrame.filter(condition)` and its alias `where(condition)` both return a new DataFrame containing the matching rows.

The examples demonstrate common conditions:

- `F.col("salary") > 85000` keeps employees whose salary exceeds the threshold.
- `(department == "Data") & (age >= 30)` combines two conditions with logical AND. Parentheses matter because Python operator precedence differs from SQL; use `&` for AND, `|` for OR, and `~` for NOT rather than Python's `and`, `or`, and `not`.
- `isin("Data", "BI")` tests whether a column's value is in a set of choices.
- `isNull()` checks for a missing Spark SQL value. Comparing to `None` with `==` does not correctly test SQL nulls because null represents unknown.

Rows where a condition is null are not considered a match. Use `isNotNull()` when you need to retain only populated values.

In [ ]:
employees.filter(F.col("salary") > 85000).show()
employees.filter((F.col("department") == "Data") & (F.col("age") >= 30)).show()
employees.filter(F.col("department").isin("Data", "BI")).show()
employees.filter(F.col("department").isNull()).show()


## Renaming columns with withColumnRenamed

`withColumnRenamed(existing, new)` returns a new DataFrame with a column label changed. It leaves the underlying values unchanged.

The example renames `join_date` to the clearer name `hire_date`, stores the returned DataFrame in `renamed`, then selects and displays the employee name with the renamed date. The original `employees` DataFrame still has its `join_date` column because DataFrames are immutable: transformations describe a new result rather than changing an existing one.

Use `withColumnRenamed()` for a simple rename. Use `withColumn()` to add a calculated column or replace a column's values.

In [ ]:
renamed = employees.withColumnRenamed("join_date", "hire_date")
renamed.select("name", "hire_date").show()


## Adding and replacing columns with withColumn

`withColumn(column_name, expression)` returns a DataFrame with a new column, or replaces the named column if it already exists. The expression is represented as a Spark `Column`, allowing Spark to plan and execute the calculation across partitions.

In this example, the first call copies `salary` into a new `annual_salary` column. The next call builds `salary_band`: `F.when()` tests the highest salary threshold first, the chained `.when()` checks the next threshold for rows that did not match earlier, and `.otherwise("entry")` supplies the fallback. The order matters because the first true condition wins.

Finally, `select()` narrows the displayed result to the employee's name, salary copy, and assigned band. The source DataFrame remains unchanged.

In [ ]:
with_band = (
    employees
    .withColumn("annual_salary", F.col("salary"))
    .withColumn(
        "salary_band",
        F.when(F.col("salary") >= 100000, "high")
        .when(F.col("salary") >= 80000, "mid")
        .otherwise("entry"),
    )
)
with_band.select("name", "annual_salary", "salary_band").show()


## Type casting

Casting converts a column's values to a different Spark SQL type. This is common when source files store everything as text but calculations require numeric or date types.

`F.col("id").cast("string")` converts the integer ID to a string and saves it as a separate demonstration column. `F.col("salary").cast(T.IntegerType())` converts the salary to an integer type; the fractional portion is not retained. `withColumn()` creates both results, and `select()` displays each original beside its converted value for comparison.

Conversion can fail for values that do not fit the target type. Depending on the type, Spark version, and ANSI settings, invalid conversions may yield null or raise an error. Check conversion results and handle bad input explicitly rather than assuming every source value is valid.

In [ ]:
casted = (
    employees
    .withColumn("id_as_string", F.col("id").cast("string"))
    .withColumn("salary_as_integer", F.col("salary").cast(T.IntegerType()))
)
casted.select("id", "id_as_string", "salary", "salary_as_integer").show()


## Sorting

`orderBy()` returns a DataFrame sorted by one or more expressions; `sort()` is an alias. Sorting is different from filtering: it changes row order, not which rows are present.

- The first expression sorts salaries from highest to lowest using `desc()`.
- The second sorts departments alphabetically, places null departments last with `asc_nulls_last()`, and uses employee name as a secondary key.
- The third explicitly places missing salaries last while sorting salaries descending.

Explicit null ordering is useful because null placement can otherwise be easy to overlook. A DataFrame has no guaranteed order unless an ordering is requested, and a sort may require Spark to shuffle data, so use it when ordering is needed rather than as a cosmetic step in a large pipeline.

In [ ]:
employees.orderBy(F.col("salary").desc()).show()
employees.orderBy(F.col("department").asc_nulls_last(), F.col("name").asc()).show()
employees.orderBy(F.col("salary").desc_nulls_last()).show()


## LIMIT

`limit(n)` restricts a DataFrame to at most `n` rows. In the example, sorting by `id` first makes the three displayed records predictable; without `orderBy()`, Spark does not promise which rows a limit will return.

`limit()` is useful for small previews, but it should not be mistaken for random sampling or a guarantee that only three source rows will be read in every execution plan. Use sampling methods when a representative random subset is required.

In [ ]:
employees.orderBy("id").limit(3).show()


## Dropping columns

`drop()` removes the named columns from the output schema and returns a new DataFrame. Here it removes `city` and `skills`, leaving employee fields that do not include location or skill information.

`printSchema()` verifies the resulting column list and types. Dropping columns early can make downstream results simpler and reduce unnecessary data processing, especially when reading columnar formats. The original `employees` remains available unchanged.

In [ ]:
without_location = employees.drop("city", "skills")
without_location.printSchema()


## Dropping duplicate rows

`dropDuplicates()` (also called `distinct()` when considering all columns) removes duplicate records. With no subset, two rows are duplicates only when all their column values match. With a subset, duplicate identity is determined only by the listed columns.

The example builds four rows, including an exact duplicate `(1, "A")` and two different values for ID 1. The first call removes only the exact duplicate. The second treats `id` as the key, so only one row for ID 1 remains; because the rows differ in `value`, which representative is retained is not deterministic.

If you must choose a particular row per key (for example, the most recent record), first define that rule with a window ordered by timestamp, then filter to the preferred row. Do not assume `dropDuplicates(subset)` picks the latest or otherwise preferred record.

In [ ]:
duplicate_rows = spark.createDataFrame(
    [(1, "A"), (1, "A"), (1, "B"), (2, "C")],
    ["id", "value"],
)
print("All-column duplicates removed:")
duplicate_rows.dropDuplicates().show()
print("One row per id:")
duplicate_rows.dropDuplicates(["id"]).show()


## UNION

`union()` appends the rows of one DataFrame after the rows of another, like stacking tables. It matches fields by **position**, so both inputs must have the same number of columns in the same order and with compatible types.

The example creates two batches with the same `id, name` schema, then combines them. Spark's DataFrame `union()` preserves duplicate rows; this differs from SQL's `UNION` keyword, which commonly removes duplicates. Call `.distinct()` explicitly if de-duplication is required.

For reliable combination, confirm schemas before unioning. If columns may appear in different orders, use `unionByName()` instead.

In [ ]:
first_batch = spark.createDataFrame([(1, "Ada"), (2, "Bo")], ["id", "name"])
second_batch = spark.createDataFrame([(3, "Cy")], ["id", "name"])
first_batch.union(second_batch).show()


## UNION BY NAME

`unionByName()` combines rows by matching column names instead of column positions. This helps when input batches have the same fields arranged differently.

The left batch has `id` and `name`; the right batch has those columns plus `department`. `allowMissingColumns=True` tells Spark to include the additional column and fill the left batch's missing department with null. The values are aligned by field name, not by ordinal position.

This option handles absent columns, not arbitrary incompatible types. Check and, when necessary, cast corresponding columns to compatible types before combining data.

In [ ]:
left_batch = spark.createDataFrame([(1, "Ada")], ["id", "name"])
right_batch = spark.createDataFrame([(2, "Bo", "Data")], ["id", "name", "department"])
left_batch.unionByName(right_batch, allowMissingColumns=True).show()


## String functions

Spark's built-in string functions operate on `Column` expressions and can be executed within Spark's query engine. They are generally preferable to Python row-by-row string logic because they avoid Python UDF serialization overhead.

- `F.initcap("name")` capitalizes the first letter of each word for a display-friendly name.
- `F.upper("department")` converts department text to uppercase; null remains null.
- `F.concat_ws(" | ", "city", "department")` joins the supplied values with a separator, skipping null values.
- `alias()` names each calculated output column, and `show(truncate=False)` displays full output strings.

These expressions create a new projection only; the original columns are unchanged.

In [ ]:
employees.select(
    "name",
    F.initcap("name").alias("title_case"),
    F.upper("department").alias("department_upper"),
    F.concat_ws(" | ", "city", "department").alias("location_and_team"),
).show(truncate=False)


## Current date

`F.current_date()` creates a Spark SQL expression that returns the current date as a `DateType`. It is evaluated by Spark as part of a query, rather than being computed by Python when the cell is first run.

The example selects each employee's name alongside that date and aliases the expression as `today`. The date column has the same current-date value for rows evaluated in the same query. Use Spark date functions when comparing dates so the values retain date semantics rather than becoming formatted strings.

In [ ]:
employees.select(
    "name",
    F.current_date().alias("today"),
).show()


## Date add and subtract

Date arithmetic should use date values, not unparsed text. `F.to_date("join_date")` parses the ISO-style strings in the sample data and `withColumn()` adds the resulting `hire_date` column to a new DataFrame.

`F.date_add("hire_date", 30)` calculates a date 30 days after the hire date; `F.date_sub("hire_date", 7)` calculates a date seven days before it. The selected output includes the employee and original parsed date so the offsets can be checked.

If source strings use a different format, provide the appropriate parsing pattern to `to_date()`. Invalid or missing input can produce null dates.

In [ ]:
dated = employees.withColumn("hire_date", F.to_date("join_date"))
dated.select(
    "name",
    "hire_date",
    F.date_add("hire_date", 30).alias("after_30_days"),
    F.date_sub("hire_date", 7).alias("one_week_before"),
).show()


## Date difference

`F.datediff(end, start)` returns the number of calendar days between two dates, with the end date first and start date second. The example first parses `join_date` into `hire_date`, then calculates the days between that date and Spark's current date.

The result is a day count, not a duration in hours or a business-day count. Null input dates produce null results. For timestamps or elapsed hours, use timestamp-aware expressions instead.

In [ ]:
dated = employees.withColumn("hire_date", F.to_date("join_date"))
dated.select(
    "name",
    "hire_date",
    F.datediff(F.current_date(), "hire_date").alias("days_employed"),
).show()


## Date formatting

`F.date_format(date_column, pattern)` converts a date or timestamp to a string using Spark's datetime pattern symbols. The example parses `join_date` once, then produces two presentation formats: `"MMMM yyyy"` yields a month name and year, while `"yyyy-MM-dd"` yields an ISO-style date string.

Formatted output is text, so it is less suitable for subsequent date comparisons or arithmetic. Keep the `DateType` column for analysis and create formatted strings only for display or a text-based export.

In [ ]:
dated = employees.withColumn("hire_date", F.to_date("join_date"))
dated.select(
    "name",
    F.date_format("hire_date", "MMMM yyyy").alias("hire_month"),
    F.date_format("hire_date", "yyyy-MM-dd").alias("iso_date"),
).show()


## Handling null values

SQL `NULL` means a value is missing or unknown; it is not the same as an empty string, zero, or the Python value `None` outside a Spark DataFrame. Conditions involving null can themselves evaluate to unknown, so use Spark's null-aware methods.

`employees.na` provides the `DataFrameNaFunctions` helper:

- `.drop()` with default settings removes any row containing a null in any column. `count()` reports how many rows remain.
- `.drop(thresh=7)` keeps rows with at least seven non-null values; here the DataFrame has eight columns.
- `.fillna({...})` replaces nulls only in the named columns, using a value compatible with each column's type.

The example uses `"Unassigned"` for a missing department and zero for missing age or salary. Those replacements are demonstrations, not universally correct business rules: zero salary has a different meaning from unknown salary. Choose policies with domain context, and consider keeping a flag that records which values were imputed.

In [ ]:
print("Rows with any null:", employees.na.drop().count())
print("Rows with at least 7 non-null fields:", employees.na.drop(thresh=7).count())

filled = employees.na.fill(
    {"department": "Unassigned", "age": 0, "salary": 0.0}
)
filled.select("name", "department", "age", "salary").show()


## SPLIT and indexing

`F.split(column, pattern)` divides each string into an array. Its second argument is a regular expression; the comma in this example separates each employee's comma-delimited skill names. `withColumn()` adds this array as `skill_list`.

`getItem(0)` accesses the first element using zero-based indexing, so index 0 is the first skill. `F.element_at(array, 2)` uses Spark's one-based positive indexing, so index 2 is the second skill. The aliases name these results, and `truncate=False` displays the full array.

An index outside the array bounds can return null. If strings may have extra whitespace, trim individual array elements before using them.

In [ ]:
skills_array = employees.withColumn("skill_list", F.split("skills", ","))
skills_array.select(
    "name",
    "skill_list",
    F.col("skill_list").getItem(0).alias("first_skill"),
    F.element_at("skill_list", 2).alias("second_skill"),
).show(truncate=False)


## EXPLODE

An array stores multiple values inside a single row. `F.explode(array_column)` turns each array element into a separate output row. All other selected columns are repeated for each element, which changes the row count but preserves the relationship to the original employee.

The example first creates `skill_list` by splitting the comma-separated `skills` text. It then selects the employee ID and name plus the result of `explode()`, naming the emitted value `skill`. An employee with three skills therefore appears on three output rows.

Regular `explode()` omits rows for null or empty arrays. Use `explode_outer()` if such input rows must remain in the result.

In [ ]:
skills_array = employees.withColumn("skill_list", F.split("skills", ","))
employees_with_one_skill = skills_array.select(
    "id", "name", F.explode("skill_list").alias("skill")
)
employees_with_one_skill.show()


## GROUP BY

Grouping partitions records by the values in one or more key columns, then computes summaries for each group. `groupBy("department")` creates a grouped DataFrame; `.agg(...)` specifies the calculations to perform:

- `F.count("*")` counts rows in each department.
- `F.avg("salary")` calculates the mean of non-null salaries.
- `F.max("salary")` returns the largest non-null salary.
- `alias()` assigns clear names to the aggregate columns.

`orderBy("department")` sorts the resulting groups for readable output. Null departments form their own group. Grouping is a distributed operation that often shuffles data between Spark tasks, so aggregate in Spark rather than collecting all rows to Python.

In [ ]:
employees.groupBy("department").agg(
    F.count("*").alias("employee_count"),
    F.avg("salary").alias("average_salary"),
    F.max("salary").alias("top_salary"),
).orderBy("department").show()


## COLLECT_LIST

`collect_list(column)` is an aggregation that gathers values from each group into an array, retaining duplicates. `collect_set(column)` similarly gathers values but removes duplicates. Both return arrays, which can be useful for compact summaries but can grow large for high-volume groups.

The example groups employees by department, collects employee names into `employee_names`, and collects distinct cities into `distinct_cities`. Null handling follows Spark aggregate semantics. The order of array elements is not guaranteed after a distributed aggregation; do not treat the result as sorted unless you explicitly sort it.

In [ ]:
employees.groupBy("department").agg(
    F.collect_list("name").alias("employee_names"),
    F.collect_set("city").alias("distinct_cities"),
).show(truncate=False)


## PIVOT

A pivot reshapes data by turning the values of one categorical column into separate output columns. It is useful for cross-tabulated summaries such as units sold by region and product.

The example creates a small sales DataFrame with `region`, `product`, and `units`. `groupBy("region")` defines the output row groups. `.pivot("product", ["Laptop", "Phone"])` makes one output column for each listed product. `.sum("units")` specifies how multiple records for a region/product combination are combined. `show()` displays the reshaped table.

Providing the pivot values explicitly documents the desired output columns and avoids Spark having to discover distinct values first. Values not listed do not become pivot columns; check the input categories when choosing this form.

In [ ]:
sales = spark.createDataFrame(
    [
        ("North", "Laptop", 3),
        ("North", "Phone", 5),
        ("South", "Laptop", 2),
        ("South", "Phone", 4),
    ],
    ["region", "product", "units"],
)
sales.groupBy("region").pivot("product", ["Laptop", "Phone"]).sum("units").show()


## WHEN and OTHERWISE

`F.when(condition, value)` creates a conditional Spark expression similar to SQL `CASE WHEN`. Chaining `.when()` adds further cases, and `.otherwise()` defines the value for rows that match none of them.

The example assigns `"unknown"` to employees whose age is null, then classifies known ages into ranges. The order is significant: Spark tests conditions in sequence and uses the first matching branch. The final `otherwise("40 or older")` catches every remaining row. Without `otherwise()`, unmatched rows produce null.

The result is added as an `age_group` column, and the selected columns make it easy to compare original ages with their classifications.

In [ ]:
classified = employees.withColumn(
    "age_group",
    F.when(F.col("age").isNull(), "unknown")
    .when(F.col("age") < 30, "under 30")
    .when(F.col("age") < 40, "30 to 39")
    .otherwise("40 or older"),
)
classified.select("name", "age", "age_group").show()


## JOINS

A join combines columns from two DataFrames based on matching key values. Here `department_budget` is a small lookup table and `department` is the common key. Passing `on="department"` asks Spark to match equal values and emit the shared key once.

The four join modes demonstrate different row-retention rules:

- `inner` keeps only employee rows whose department has a matching budget.
- `left` keeps every employee; budget columns are null when there is no matching department.
- `left_semi` keeps only columns from the left DataFrame, and only for employees with a match. Use it as an existence filter.
- `left_anti` keeps only left-side employees with no matching budget.

In Spark SQL, null join keys do not match other nulls under ordinary equality. When joining on different column names or more complex conditions, pass an explicit expression and rename overlapping fields to avoid ambiguous references.

In [ ]:
department_budget = spark.createDataFrame(
    [("Data", 500000.0), ("BI", 250000.0)],
    ["department", "budget"],
)

employees.join(department_budget, on="department", how="inner").show()
employees.join(department_budget, on="department", how="left").show()
employees.join(department_budget, on="department", how="left_semi").show()
employees.join(department_budget, on="department", how="left_anti").show()


## ROW_NUMBER

Window functions calculate values across related rows while keeping each input row in the result. A `WindowSpec` describes which rows are related and in what order.

`Window.partitionBy("department")` creates a separate numbering group for each department. `.orderBy(F.col("salary").desc_nulls_last())` puts higher salaries first and missing salaries last. `F.row_number().over(department_window)` assigns 1, 2, 3, ... within each group, and `withColumn()` adds those numbers to a new DataFrame.

Unlike `rank()`, `row_number()` always gives each row a unique position, even where salaries tie. If tied rows need a repeatable order, add a second ordering key such as employee ID.

In [ ]:
department_window = Window.partitionBy("department").orderBy(F.col("salary").desc_nulls_last())
ranked = employees.withColumn("row_number", F.row_number().over(department_window))
ranked.select("name", "department", "salary", "row_number").show()


## RANK versus DENSE_RANK

Both functions rank rows according to a window's ordering and give the same rank to tied values. Their difference is whether a tie creates a gap in subsequent ranks.

The example sorts students by descending score. Scores are 95, 90, 90, and 80. `rank()` produces 1, 2, 2, 4 because two students share second place and the next rank is skipped. `dense_rank()` produces 1, 2, 2, 3 because its ranking has no gaps. The `.over(score_window)` call applies each function to the ordered window.

Use `rank()` when competition-style ranking is desired, and `dense_rank()` when consecutive rank labels are preferable. Add partitioning to the window when ranking separately within groups.

In [ ]:
scores = spark.createDataFrame(
    [("A", 95), ("B", 90), ("C", 90), ("D", 80)],
    ["student", "score"],
)
score_window = Window.orderBy(F.col("score").desc())
scores.select(
    "student",
    "score",
    F.rank().over(score_window).alias("rank"),
    F.dense_rank().over(score_window).alias("dense_rank"),
).show()


## Cumulative sum

A cumulative sum adds values in a defined sequence, retaining one output per input row. It is implemented as `F.sum(column).over(window)`, where the window establishes the group, ordering, and frame of rows visible to each result.

Here `partitionBy("department")` restarts the total for each department. `orderBy("join_date", "id")` defines a chronological order and uses ID to break same-date ties. `rowsBetween(Window.unboundedPreceding, Window.currentRow)` includes every row from the beginning of that department's ordered partition through the current row.

The resulting `running_salary_total` is a running sum of salary. A deterministic order matters: without it, the sequence of accumulated values is undefined. Rows with null salary do not contribute to `sum`; null department values are treated as their own partition.

In [ ]:
running_window = (
    Window.partitionBy("department")
    .orderBy("join_date", "id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
employees.withColumn(
    "running_salary_total",
    F.sum("salary").over(running_window),
).select("name", "department", "join_date", "salary", "running_salary_total").show()


## User-defined functions (UDF)

A Python UDF lets you apply custom Python logic to a Spark column when a built-in Spark expression is not sufficient. The `@F.udf(returnType=T.StringType())` decorator registers the Python function as a Spark UDF and tells Spark that its output values are strings.

`normalize_name(value)` returns a title-cased string for a non-null input and preserves missing values as `None` (Spark SQL null). Calling `normalize_name("name")` with a column name creates a Spark expression; it does not invoke the function once immediately on the string `"name"`. The selected result is displayed next to the original.

Python UDFs can be slower than built-in expressions because values cross the JVM/Python boundary and Spark has less opportunity to optimize the function's internal logic. For this particular transformation, a built-in such as `F.initcap("name")` is preferable. The UDF is included to teach the interface; use it when the required logic genuinely cannot be expressed with native functions.

In [ ]:
@F.udf(returnType=T.StringType())
def normalize_name(value):
    return value.title() if value is not None else None

employees.select(
    "name",
    normalize_name("name").alias("normalized_name"),
).show()


## Writing CSV and save modes

Writing is an action: Spark evaluates the DataFrame and stores output at a destination. CSV is a text format, so it does not preserve rich types as reliably as Parquet. Spark generally creates a **directory** containing one or more part files rather than one file with the requested directory name.

The example uses `tempfile.TemporaryDirectory()` so the output is isolated and automatically cleaned up after the block. `Path(temp_dir) / "employees_csv"` constructs a platform-appropriate destination. The selected columns control which data is written.

`write.mode(...)` chooses behavior if the destination already exists. `option("header", True)` writes a header line for each CSV part. The first write uses `overwrite`; the next writes one more row using `append`. Reading back with `spark.read.option("header", True).csv(...)` interprets the first line of each part as column names. CSV reading commonly infers text columns unless a schema is supplied.

The second destination demonstrates `errorifexists`, which creates a fresh path successfully, and then `ignore`, which leaves an existing path unchanged. If the first destination for `errorifexists` already existed, Spark would raise an error; this is why the example uses a temporary fresh path. Other modes include `error` (an alias for `errorifexists`) and `ignore`. Avoid appending CSV data with inconsistent schemas or header conventions.

In [ ]:
with tempfile.TemporaryDirectory() as temp_dir:
    output = Path(temp_dir) / "employees_csv"
    employees.select("id", "name", "department").write.mode("overwrite").option(
        "header", True
    ).csv(str(output))

    # Append another small batch to the same output directory.
    spark.createDataFrame([(7, "gail white", "BI")], employees.select(
        "id", "name", "department"
    ).schema).write.mode("append").option("header", True).csv(str(output))
    spark.read.option("header", True).csv(str(output)).show()

    # A fresh output path demonstrates error-if-exists without raising.
    error_output = Path(temp_dir) / "error_if_exists_example"
    employees.limit(1).write.mode("errorifexists").csv(str(error_output))

    # The existing destination is left unchanged by ignore.
    employees.limit(1).write.mode("ignore").csv(str(error_output))


## Writing Parquet

Parquet is a columnar file format designed for analytical workloads. Unlike CSV, it stores schema and types and commonly supports compression, column pruning (reading only requested columns), and predicate pushdown (skipping irrelevant data where possible).

The example writes `employees` to a temporary Parquet directory with overwrite mode, then reads it back with `spark.read.parquet(path)`. `printSchema()` verifies that types such as integer, double, and string were retained. The final `select()` reads only a few columns for display.

Temporary output is cleaned up when the `with` block ends. For real projects, choose a durable path and an intentional save mode; writing Parquet is generally more suitable than CSV for data that Spark will analyze again.

In [ ]:
with tempfile.TemporaryDirectory() as temp_dir:
    parquet_path = Path(temp_dir) / "employees_parquet"
    employees.write.mode("overwrite").parquet(str(parquet_path))
    parquet_copy = spark.read.parquet(str(parquet_path))
    parquet_copy.printSchema()
    parquet_copy.select("name", "department", "salary").show()


## Writing a table

`saveAsTable(name)` stores a DataFrame as a table registered in Spark's catalog. Unlike writing to an arbitrary file path, later queries can refer to the table by its catalog name. The exact storage location and durability depend on the configured catalog and Spark environment.

The example uses a fixed table name and `mode("overwrite")`, so rerunning the cell replaces the prior contents under that name. `spark.table(table_name)` loads the catalog table as a DataFrame, and `select(...).show()` queries a few fields.

Use a unique name or a temporary table for experiments in a shared catalog, and remove the table when it is no longer needed. A catalog table is distinct from a temporary view, which exists only within a Spark session and does not itself persist the data.

In [ ]:
table_name = "pyspark_fundamentals_employees"
employees.write.mode("overwrite").saveAsTable(table_name)
spark.table(table_name).select("name", "department", "salary").show()


## Spark SQL

Spark SQL is Spark's SQL interface for querying structured data. A DataFrame can be registered under a view name and then referenced in SQL statements.

`employees.createOrReplaceTempView("employees_view")` registers a temporary view in the current Spark session. It does not copy or write the data; the view points to the DataFrame's query plan and is available only while that session remains active.

`spark.sql("""...""")` parses and executes a SQL query and returns its result as another DataFrame. The query filters out null salaries, groups by department, calculates a row count and rounded average salary, and sorts the aggregate results by descending average. `show()` triggers execution and displays the result.

Use backticks around identifiers that contain spaces or conflict with SQL keywords. Temporary views are useful for mixing the DataFrame API and SQL while keeping the same underlying data.

In [ ]:
employees.createOrReplaceTempView("employees_view")
spark.sql(
    """
    SELECT department, COUNT(*) AS employee_count, ROUND(AVG(salary), 2) AS avg_salary
    FROM employees_view
    WHERE salary IS NOT NULL
    GROUP BY department
    ORDER BY avg_salary DESC
    """
).show()
